# Day 13 — Validate the real Olist CSV files

In [1]:
## 1. Configuration and required fields

import csv, json, re
from datetime import datetime
from pathlib import Path

DATA_DIR = Path("olist")
BAD_RECORDS_PATH = Path("bad_records.csv")
REJECTION_LIMIT = 0.20 
ALLOWED_ORDER_STATUSES = {"created", "approved", "invoiced", "processing", "shipped", "delivered", "unavailable", "canceled"}

SCHEMAS = {
 "customers.csv": (",", {"customer_id", "customer_unique_id", "customer_zip_code_prefix", "customer_city", "customer_state"}),
 "geolocation.csv": (",", {"geolocation_zip_code_prefix", "geolocation_lat", "geolocation_lng", "geolocation_city", "geolocation_state"}),
 "orders.csv": (",", {"order_id", "customer_id", "order_status", "order_purchase_timestamp", "order_approved_at", "order_delivered_carrier_date", "order_delivered_customer_date", "order_estimated_delivery_date"}),
 "orders_semicolon.csv": (";", {"order_id", "customer_id", "order_status", "order_purchase_timestamp", "order_approved_at", "order_delivered_carrier_date", "order_delivered_customer_date", "order_estimated_delivery_date"}),
 "order_items.csv": (",", {"order_id", "order_item_id", "product_id", "seller_id", "shipping_limit_date", "price", "freight_value"}),
 "payments.csv": (",", {"order_id", "payment_sequential", "payment_type", "payment_installments", "payment_value"}),
 "products.csv": (",", {"product_id", "product_category_name", "product_name_lenght", "product_description_lenght", "product_photos_qty", "product_weight_g", "product_length_cm", "product_height_cm", "product_width_cm"}),
 "product_category_name_translation.csv": (",", {"product_category_name", "product_category_name_english"}),
 "reviews.csv": (",", {"review_id", "order_id", "review_score", "review_comment_title", "review_comment_message", "review_creation_date", "review_answer_timestamp"}),
 "sellers.csv": (",", {"seller_id", "seller_zip_code_prefix", "seller_city", "seller_state"}),
}
REQUIRED_VALUES = {
 "customers.csv": {"customer_id", "customer_unique_id", "customer_zip_code_prefix"},
 "geolocation.csv": {"geolocation_zip_code_prefix"},
 "orders.csv": {"order_id", "customer_id", "order_status", "order_purchase_timestamp"},
 "orders_semicolon.csv": {"order_id", "customer_id", "order_status", "order_purchase_timestamp"},
 "order_items.csv": {"order_id", "order_item_id", "product_id", "seller_id", "shipping_limit_date", "price", "freight_value"},
 "payments.csv": {"order_id", "payment_sequential", "payment_type", "payment_installments", "payment_value"},
 "products.csv": {"product_id"},
 "product_category_name_translation.csv": {"product_category_name", "product_category_name_english"},
 "reviews.csv": {"review_id", "order_id", "review_score", "review_creation_date"},
 "sellers.csv": {"seller_id", "seller_zip_code_prefix"},
}
DATE_COLUMNS = {"order_purchase_timestamp", "order_approved_at", "order_delivered_carrier_date", "order_delivered_customer_date", "order_estimated_delivery_date", "shipping_limit_date", "review_creation_date", "review_answer_timestamp"}
print("Configured", len(SCHEMAS), "Olist CSVs")

Configured 10 Olist CSVs


In [2]:
## 2. Custom exceptions


class SchemaError(Exception):
    pass

class FileFormatError(Exception):
    pass

print("SchemaError and FileFormatError are ready")

SchemaError and FileFormatError are ready


In [6]:
## 3. Regex patterns and validating one row


ID_PATTERN = re.compile(r"^[0-9a-fA-F]{32}$")
ZIP_PATTERN = re.compile(r"^\d{5}$")
NUMERIC_COLUMNS = {"price", "freight_value", "payment_value", "geolocation_lat", "geolocation_lng", "product_weight_g", "product_length_cm", "product_height_cm", "product_width_cm", "review_score", "payment_sequential", "payment_installments", "order_item_id", "product_name_lenght", "product_description_lenght", "product_photos_qty"}

def validate_row(row):
    reasons = []
    for column, raw in row.items():
        value = (raw or "").strip()
        if not value:  
            continue
        if column.endswith("_id") and column != "order_item_id" and not ID_PATTERN.fullmatch(value):
            reasons.append(f"{column} must be 32 hexadecimal characters")
        if column.endswith("zip_code_prefix") and not ZIP_PATTERN.fullmatch(value):
            reasons.append(f"{column} must be exactly 5 digits")
        if column in NUMERIC_COLUMNS:
            try:
                number = float(value)
                if column == "price" and number <= 0:
                    reasons.append("price must be greater than 0")
            except ValueError:
                reasons.append(f"{column} must be numeric: {value!r}")
        if column in DATE_COLUMNS:
            try:
                datetime.fromisoformat(value)
            except ValueError:
                reasons.append(f"{column} is not a parseable date/time: {value!r}")
    status = (row.get("order_status") or "").strip()
    if status and status not in ALLOWED_ORDER_STATUSES:
        reasons.append(f"order_status is not allowed: {status!r}")
    return reasons

print(validate_row({"price": "25.50", "order_status": "delivered", "customer_zip_code_prefix": "01001"}))
print(validate_row({"price": "0", "order_status": "flying", "customer_zip_code_prefix": "12A"}))

[]
['price must be greater than 0', 'customer_zip_code_prefix must be exactly 5 digits', "order_status is not allowed: 'flying'"]


In [7]:
## 4. Read and validate every row


def validate_file(path, delimiter, required_columns):
    summary = {"file": path.name, "total": 0, "valid": 0, "rejected": 0, "status": "ok"}
    bad_rows = []
    if not path.exists():
        summary["status"] = "missing (skipped)"
        print(path.name, "is missing; skipped")
        return summary, bad_rows
    if path.stat().st_size == 0:
        summary["status"] = "empty (skipped)"
        print(path.name, "is empty; skipped")
        return summary, bad_rows
    try:
        with path.open(encoding="utf-8-sig", newline="") as file:
            reader = csv.DictReader(file, delimiter=delimiter)
            headers = reader.fieldnames or []
            if len(headers) == 1 and delimiter not in headers[0]:
                raise FileFormatError(f"Expected delimiter {delimiter!r}; header appears combined")
            if len(headers) != len(set(headers)):
                raise SchemaError("Duplicate column names")
            missing = required_columns - set(headers)
            if missing:
                raise SchemaError(f"Missing required columns: {sorted(missing)}")
            for line_number, row in enumerate(reader, start=2):
                summary["total"] += 1
                reasons = (["row has more values than the header"] if None in row else validate_row(row))
                if reasons:
                    summary["rejected"] += 1
                    bad_rows.append({"source_file": path.name, "row_number": line_number,
                                     "reason": "; ".join(reasons),
                                     "raw_row": json.dumps(row, ensure_ascii=False)})
                else:
                    summary["valid"] += 1
        if summary["total"] == 0:
            summary["status"] = "no data rows (skipped)"
        elif summary["rejected"] / summary["total"] > REJECTION_LIMIT:
            summary["status"] = "failed: rejection limit exceeded"
    except (SchemaError, FileFormatError, UnicodeDecodeError, csv.Error) as error:
        summary["status"] = f"file error: {error}"
    except OSError as error:
        summary["status"] = f"could not read file: {error}"
    finally:
        print(f"Finished checking {path.name}: {summary['status']}")
    return summary, bad_rows

In [8]:
## 5. Run the rules, write rejects, and summarize


summaries, all_bad_rows = [], []
for filename, (delimiter, required) in SCHEMAS.items():
    summary, bad_rows = validate_file(DATA_DIR / filename, delimiter, required)
    summaries.append(summary)
    all_bad_rows.extend(bad_rows)

with BAD_RECORDS_PATH.open("w", encoding="utf-8", newline="") as file:
    fields = ["source_file", "row_number", "reason", "raw_row"]
    writer = csv.DictWriter(file, fieldnames=fields)
    writer.writeheader()
    writer.writerows(all_bad_rows)

print("\nSUMMARY: total, valid, rejected rows")
for item in summaries:
    print(f"{item['file']}: total={item['total']}, valid={item['valid']}, rejected={item['rejected']}, status={item['status']}")
print("ALL FILES:", sum(x["total"] for x in summaries), "total,",
      sum(x["valid"] for x in summaries), "valid,",
      sum(x["rejected"] for x in summaries), "rejected")
print("Rejected rows saved to", BAD_RECORDS_PATH.resolve())

Finished checking customers.csv: ok
Finished checking geolocation.csv: ok
Finished checking orders.csv: ok
Finished checking orders_semicolon.csv: ok
Finished checking order_items.csv: ok
Finished checking payments.csv: ok
Finished checking products.csv: ok
Finished checking product_category_name_translation.csv: ok
Finished checking reviews.csv: ok
Finished checking sellers.csv: ok

SUMMARY: total, valid, rejected rows
customers.csv: total=99441, valid=99441, rejected=0, status=ok
geolocation.csv: total=1000163, valid=1000163, rejected=0, status=ok
orders.csv: total=99441, valid=99441, rejected=0, status=ok
orders_semicolon.csv: total=99441, valid=99441, rejected=0, status=ok
order_items.csv: total=112650, valid=112650, rejected=0, status=ok
payments.csv: total=103886, valid=103886, rejected=0, status=ok
products.csv: total=32951, valid=32951, rejected=0, status=ok
product_category_name_translation.csv: total=71, valid=71, rejected=0, status=ok
reviews.csv: total=99224, valid=99224, r

In [9]:
failed_files = [
    item["file"]
    for item in summaries
    if item["status"] == "failed: rejection limit exceeded"
]

if failed_files:
    raise FileFormatError(
        f"Whole-file validation failed for {failed_files}. "
        f"See {BAD_RECORDS_PATH} for rejected rows."
    )
else:
    print(f"No file exceeded the {REJECTION_LIMIT:.0%} rejection limit.")

No file exceeded the 20% rejection limit.


In [11]:
# 7
class SchemaError(Exception):
    pass


class FileFormatError(Exception):
    pass

In [ ]:
#8
def check_file_schema(file_path, required_fields, delimiter=","):
    with open(file_path, encoding="utf-8-sig", newline="") as file:
        headers = next(csv.reader(file, delimiter=delimiter), [])

    missing = set(required_fields) - set(headers)
    if missing:
        raise SchemaError(f"Missing columns: {sorted(missing)}")

In [13]:
#9

print("\nSUMMARY")

for item in summaries:
    print(
        f"{item['file']}: "
        f"total={item['total']}, "
        f"valid={item['valid']}, "
        f"rejected={item['rejected']}, "
        f"status={item['status']}"
    )

total = sum(item["total"] for item in summaries)
valid = sum(item["valid"] for item in summaries)
rejected = sum(item["rejected"] for item in summaries)

print(f"ALL FILES: total={total}, valid={valid}, rejected={rejected}")


SUMMARY
customers.csv: total=99441, valid=99441, rejected=0, status=ok
geolocation.csv: total=1000163, valid=1000163, rejected=0, status=ok
orders.csv: total=99441, valid=99441, rejected=0, status=ok
orders_semicolon.csv: total=99441, valid=99441, rejected=0, status=ok
order_items.csv: total=112650, valid=112650, rejected=0, status=ok
payments.csv: total=103886, valid=103886, rejected=0, status=ok
products.csv: total=32951, valid=32951, rejected=0, status=ok
product_category_name_translation.csv: total=71, valid=71, rejected=0, status=ok
reviews.csv: total=99224, valid=99224, rejected=0, status=ok
sellers.csv: total=3095, valid=3095, rejected=0, status=ok
ALL FILES: total=1650363, valid=1650363, rejected=0


In [14]:
#10

failed_files = [
    item["file"]
    for item in summaries
    if item["status"] == "failed: rejection limit exceeded"
]

if failed_files:
    raise FileFormatError(
        f"Whole-file validation failed for {failed_files}. "
        f"See {BAD_RECORDS_PATH} for rejected rows."
    )
else:
    print(f"No file exceeded the {REJECTION_LIMIT:.0%} rejection limit.")

No file exceeded the 20% rejection limit.
